# Try-Catch Except 문

In [6]:
try:  # db연동 -> 0은 데이터베이스 연동이 끊긴 상태로 봄 
      # 에러 났지만 무시하고 코드를 짜야하는 경우에 사용 
    numerator = 10
    denominator = 2
    result = numerator / denominator
except ZeroDivisionError:
    print("Error : Cannot Divide By Zero!")
else:
    print("The result is:", result)
finally:
    print("THis will always excute.")
    a = 1
    b = 2
    print(a+b)

The result is: 5.0
THis will always excute.
3


In [9]:
try:
    numerator = 10
    denominator = int(input("숫자를 입력해주세요: "))
    result = numerator / denominator
except ZeroDivisionError:
    print("Error : Cannot Divide By Zero!")
else:
    print("The result is:", result)
finally:
    print("THis will always excute.")
    a = 1
    b = 2
    print(a+b)

숫자를 입력해주세요:  0


Error : Cannot Divide By Zero!
THis will always excute.
3


# Step1. Connect to Server
- MySQL localhost 서버와 연결하는 코드 

In [13]:
from mysql import connector
PASSWORD = '12345678'
PASSWORD = input("DB Password를 입력해주세요...")

try:
    #Database 연동하는 코드 작성
    with connector.connect(
        host = 'localhost', #회사에서는 ip 주소 입력
        user = 'root',      #회사에서 아이디 부여 
        password = PASSWORD
    ) as database:
        print(f"데이터베이스 객체(object): {database}")
except connector.Error as e:
    print(e)
finally:
    print("무조건 실행")

DB Password를 입력해주세요... 2


1045 (28000): Access denied for user 'root'@'localhost' (using password: YES)
무조건 실행


# Step2. 데이터베이스 생성

In [14]:
PASSWORD = "12345678"

try:
    #Database 연동하는 코드 작성
    with connector.connect(
        host = 'localhost', #회사에서는 ip 주소 입력
        user = 'root',      #회사에서 아이디 부여 
        password = PASSWORD
    ) as database:
        print(f"데이터베이스 객체(object): {database}")
        print("DATABASE 생성-----")
        delete_db = "DROP DATABASE book_ratings" # SQL 코드 작성
        create_db = "CREATE DATABASE book_ratings;" 

        with database.cursor() as cursor:
            cursor.execute(create_db)

            # 현재 점속한 서버의 데이터베이스를 확인 
            show_existing_db = "SHOW DATABASES;"
            cursor.execute(show_existing_db)
            # 반복문 활용
            for db in cursor:
                print(db)


except connector.Error as e:
    print(e)
finally:
    print("무조건 실행")

데이터베이스 객체(object): <mysql.connector.connection_cext.CMySQLConnection object at 0x000001E018ACCB30>
DATABASE 생성-----
('book_ratings',)
('classicmodels',)
('dataset2',)
('etc',)
('information_schema',)
('instacart',)
('mysql',)
('performance_schema',)
('sakila',)
('sys',)
('titanic',)
('world',)
무조건 실행


## SQL 서버 + DB에 직접 접속하는 방법 

In [15]:
PASSWORD = "12345678"
DATABASE_NAME = 'book_ratings'

try:
    #Database 연동하는 코드 작성
    with connector.connect(
        host = 'localhost', #회사에서는 ip 주소 입력
        user = 'root',      #회사에서 아이디 부여 
        password = PASSWORD,
        database = DATABASE_NAME
    ) as database:
        print(f"데이터베이스 객체(object): {database}")

except connector.Error as e:
    print(e)
finally:
    print("무조건 실행")

데이터베이스 객체(object): <mysql.connector.connection_cext.CMySQLConnection object at 0x000001E018A594F0>
무조건 실행


# Step3. 테이블 생성 
- 쿼리가 길어지면 multi-line 문자열로 저장

In [16]:
create_books_table = """
CREATE TABLE books(
    id INT NOT NULL AUTO_INCREMENT PRIMARY KEY,
    title VARCHAR(100),
    author VARCHAR(100),
    genre VARCHAR(100),
    release_year YEAR(4)
)
"""

create_books_table

'\nCREATE TABLE books(\n    id INT NOT NULL AUTO_INCREMENT PRIMARY KEY,\n    title VARCHAR(100),\n    author VARCHAR(100),\n    genre VARCHAR(100),\n    release_year YEAR(4)\n)\n'

In [18]:
PASSWORD = "12345678"
DATABASE_NAME = 'book_ratings'

try:
    #Database 연동하는 코드 작성
    with connector.connect(
        host = 'localhost', #회사에서는 ip 주소 입력
        user = 'root',      #회사에서 아이디 부여 
        password = PASSWORD,
        database = DATABASE_NAME
    ) as database:
        print(f"데이터베이스 객체(object): {database}")
        print("DATABASE 생성-----")

        with database.cursor() as cursor:
            # cursor.execute(create_books_table) # 테이블 생성 코드 
            # 커밋
            database.commit() # MySQL 서버에 반영함 

            # 쿼리, DESCRIBE books
            query1 = "DESCRIBE books;"
            # query2 = "SELECT * FROM orders;"
            cursor.execute(query1)
            books_schema = cursor.fetchall()
            for column in books_schema:
                print(column)

except connector.Error as e:
    print(e)
finally:
    print("무조건 실행")

데이터베이스 객체(object): <mysql.connector.connection_cext.CMySQLConnection object at 0x000001E01892C740>
DATABASE 생성-----
('id', 'int', 'NO', 'PRI', None, 'auto_increment')
('title', 'varchar(100)', 'YES', '', None, '')
('author', 'varchar(100)', 'YES', '', None, '')
('genre', 'varchar(100)', 'YES', '', None, '')
('release_year', 'year', 'YES', '', None, '')
무조건 실행


# Step4. 데이터 추가 

In [19]:
# INSERT SINGLE RECORD
insert_single_record = "INSERT INTO books (id, title, author, genre, release_year)\
    VALUES (%s, %s, %s, %s, %s)"
single_record = (
    "1", "Choose Yourself! Be Happy, Make Millions, Live the Dream", "James Altucher", "self-help", "2013"
    )

try: 
    # Connect to existing database
    with connector.connect(
        host = "localhost",
        user = "root",
        password = PASSWORD,
        database = "book_ratings"
    ) as existing_database:
        
        # Create cursor object
        with existing_database.cursor() as cursor:
            cursor.execute(insert_single_record, single_record)
            existing_database.commit()
        
except connector.Error as e: 
    print(e)

# INSERT MULTIPLE RECORDS
insert_multiple_records = "INSERT INTO books (id, title, author, genre, release_year)\
    VALUES (%s, %s, %s, %s, %s)"
multiple_records = [
    (
        "2", 
        "Skip the Line: The 10,000 Experiments Rule and Other Surprising Advice for Reaching Your Goals",
        "James Altucher",
        "self-help",
        "2021"        
    ),
    (
        "3",
        "The Power of No: Because One Little Word Can Bring Health, Abundance, and Happiness",
        "James Altucher",
        "self-help",
        "2014"
    ),
    (
        "4",
        "The 48 Laws of Power",
        "Robert Greene",
        "self-help",
        "1998"
    ),
    (
        "5",
        "Mastery",
        "Robert Greene",
        "self-help",
        "2012"
    ),
    (
        "6",
        "The Art of Seduction",
        "Robert Greene",
        "self-help",
        "2001"
    ),
]

try: 
    # Connect to existing database
    with connector.connect(
        host = "localhost",
        user = "root",
        password = PASSWORD,
        database = "book_ratings"
    ) as existing_database:
        
        # Create cursor object
        with existing_database.cursor() as cursor:
            cursor.executemany(insert_multiple_records, multiple_records)
            existing_database.commit()
        
except connector.Error as e: 
    print(e)

# Step5. Select 데이터 

In [20]:
PASSWORD = "12345678"
DATABASE_NAME = 'book_ratings'

try:
    #Database 연동하는 코드 작성
    with connector.connect(
        host = 'localhost', #회사에서는 ip 주소 입력
        user = 'root',      #회사에서 아이디 부여 
        password = PASSWORD,
        database = DATABASE_NAME
    ) as database:
        print(f"데이터베이스 객체(object): {database}")
        print("DATABASE 생성-----")

        with database.cursor() as cursor:
            # cursor.execute(create_books_table) # 테이블 생성 코드 
            # 커밋
            database.commit() # MySQL 서버에 반영함 

            # 쿼리, DESCRIBE books
            query1 = "SELECT * FROM books;"
            cursor.execute(query1)
            books_schema = cursor.fetchall()
            for column in books_schema:
                print(column)

except connector.Error as e:
    print(e)
finally:
    print("무조건 실행")

데이터베이스 객체(object): <mysql.connector.connection_cext.CMySQLConnection object at 0x000001E018ADEC90>
DATABASE 생성-----
(1, 'Choose Yourself! Be Happy, Make Millions, Live the Dream', 'James Altucher', 'self-help', 2013)
(2, 'Skip the Line: The 10,000 Experiments Rule and Other Surprising Advice for Reaching Your Goals', 'James Altucher', 'self-help', 2021)
(3, 'The Power of No: Because One Little Word Can Bring Health, Abundance, and Happiness', 'James Altucher', 'self-help', 2014)
(4, 'The 48 Laws of Power', 'Robert Greene', 'self-help', 1998)
(5, 'Mastery', 'Robert Greene', 'self-help', 2012)
(6, 'The Art of Seduction', 'Robert Greene', 'self-help', 2001)
무조건 실행


# Step6. pandas 데이터프레임으로 데이터 가져오기 

In [25]:
import mysql.connector as connector
import pandas as pd

PASSWORD = '12345678'  
def fetch_data_to_dataframe(query, database=None):
    try: 

        with connector.connect(
            host="localhost",
            user="root",
            password=PASSWORD,
            database=database
        ) as existing_database:
            
            with existing_database.cursor() as cursor:
                cursor.execute(query)

                # 테이블의 컬럼명만 추출하는 코드 => List 형태로 변형 
                column_names = [i[0] for i in cursor.description]
                
                returned_data = cursor.fetchall()
                
                df = pd.DataFrame(returned_data, columns=column_names)
                return df
    
    except connector.Error as e: 
        print(e)
        return None

query_specific_cols = "SELECT author, release_year FROM books"
df_specific_cols = fetch_data_to_dataframe(query_specific_cols, "book_ratings")
if df_specific_cols is not None:
    print("DataFrame with specific columns:")
    print(df_specific_cols)

query_all_cols = "SELECT * FROM books"
df_all_cols = fetch_data_to_dataframe(query_all_cols, "book_ratings")
if df_all_cols is not None:
    print("DataFrame with all columns:")
    print(df_all_cols)

DataFrame with specific columns:
           author  release_year
0  James Altucher          2013
1  James Altucher          2021
2  James Altucher          2014
3   Robert Greene          1998
4   Robert Greene          2012
5   Robert Greene          2001
DataFrame with all columns:
   id                                              title          author  \
0   1  Choose Yourself! Be Happy, Make Millions, Live...  James Altucher   
1   2  Skip the Line: The 10,000 Experiments Rule and...  James Altucher   
2   3  The Power of No: Because One Little Word Can B...  James Altucher   
3   4                               The 48 Laws of Power   Robert Greene   
4   5                                            Mastery   Robert Greene   
5   6                               The Art of Seduction   Robert Greene   

       genre  release_year  
0  self-help          2013  
1  self-help          2021  
2  self-help          2014  
3  self-help          1998  
4  self-help          2012  
5  self-

In [22]:
df_specific_cols

,author,release_year
0,James Altucher,2013
1,James Altucher,2021
2,James Altucher,2014
3,Robert Greene,1998
4,Robert Greene,2012
5,Robert Greene,2001


In [23]:
query_specific_cols = "SELECT * FROM orders"
orders = fetch_data_to_dataframe(query_specific_cols, "classicmodels")
orders

,orderNumber,orderDate,requiredDate,shippedDate,status,comments,customerNumber
0,10100,2003-01-06,2003-01-13,2003-01-10,Shipped,None,363
1,10101,2003-01-09,2003-01-18,2003-01-11,Shipped,Check on availability.,128
2,10102,2003-01-10,2003-01-18,2003-01-14,Shipped,None,181
3,10103,2003-01-29,2003-02-07,2003-02-02,Shipped,None,121
4,10104,2003-01-31,2003-02-09,2003-02-01,Shipped,None,141
...,...,...,...,...,...,...,...
321,10421,2005-05-29,2005-06-06,None,In Process,Custom shipping instructions were sent to ware...,124
322,10422,2005-05-30,2005-06-11,None,In Process,None,157
323,10423,2005-05-30,2005-06-05,None,In Process,None,314
324,10424,2005-05-31,2005-06-08,None,In Process,None,141


In [26]:
query_specific_cols = """
SELECT 
	MAX(주문건수), MIN(주문건수), AVG(주문건수) AS 평균주문건수
FROM (
	SELECT 
		ordernumber, count(ordernumber) AS 주문건수
	FROM orderdetails
	GROUP BY ordernumber
) A
;
"""

orders = fetch_data_to_dataframe(query_specific_cols, "classicmodels")
orders

,MAX(주문건수),MIN(주문건수),평균주문건수
0,18,1,9.1902
